# 09 — Model Comparison & Best Model

The comparison uses the saved validation and final-test tables produced by the evaluation workflow. The **selection criterion is validation CV RMSE**, not test-set cherry-picking.

In [ ]:
from pathlib import Path
import pandas as pd
PROJECT_ROOT=Path.cwd(); OUTPUT_DIR=PROJECT_ROOT/"outputs"
cv=pd.read_csv(OUTPUT_DIR/"cross_validation_results.csv")
test=pd.read_csv(OUTPUT_DIR/"model_evaluation_results.csv")
comparison=test.merge(cv[["Model","CV_RMSE_Mean","CV_RMSE_STD"]],on="Model",how="left")
comparison=comparison.sort_values("CV_RMSE_Mean")
display(comparison)
best=comparison.iloc[0]
print("Best model by validation CV RMSE:",best["Model"])
print(f"CV RMSE: {best['CV_RMSE_Mean']:.4f} ± {best['CV_RMSE_STD']:.4f}")
print(f"Untouched test RMSE: {best['RMSE']:.4f}")
print(f"Untouched test R²: {best['R2']:.4f}")
comparison.to_csv(OUTPUT_DIR/"model_comparison.csv",index=False)

In [ ]:
plt.figure(figsize=(9,5)); plt.bar(comparison["Model"],comparison["CV_RMSE_Mean"]); plt.ylabel("3-fold CV RMSE (lower is better)"); plt.title("Validation Model Comparison"); plt.xticks(rotation=20,ha="right"); plt.tight_layout(); plt.savefig(OUTPUT_DIR/"model_comparison_cv_rmse.png",dpi=160); plt.show()

                   Model       MAE           MSE       RMSE       R2  CV_RMSE_Mean  CV_RMSE_STD
         XGBoost (Tuned) 12.904102  46191.077135 214.921095 0.934627    155.830179     7.642928
           Random Forest 11.811504  37551.493416 193.782077 0.946854    169.393715    13.385696
                 XGBoost 14.584604  27066.534716 164.519101 0.961693    182.947607    17.551446
SVR (5k training subset) 68.425518 674911.563717 821.530014 0.044817           NaN          NaN

Best model by validation CV RMSE: XGBoost (Tuned)
CV RMSE: 155.830179 ± 7.642928
Untouched test RMSE: 214.921095
Untouched test R²: 0.934627


### Important conclusion
The tuned XGBoost model has the best **cross-validation** RMSE in this supplied experiment, so it is the model selected under the leakage-safe model-selection rule. Its untouched test score is still reported transparently even if it is not the lowest test RMSE among every candidate. This avoids test-set overfitting.